In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/dog005/dataset.json


In [2]:
cleaned_data = []
index = 1723

In [3]:
!pip install openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.3/328.3 kB 5.8 MB/s eta 0:00:00a 0:00:01


In [4]:
import json
import openai  # 确保你已经安装了 openai 包
import re
# 设置OpenAI API密钥
client = openai.OpenAI(api_key='sk-1be51a06f08c4073b77929d03ea358be', base_url="https://dashscope.aliyuncs.com/compatible-mode/v1")


# 第一步：读取原始JSON数据
with open('/kaggle/input/dog005/dataset.json', 'r', encoding='utf-8') as file:
    data = json.load(file)
QA_PAIRS_SYSTEM_PROMPT = """
<Context></Context> 标记中是一条用于进行大模型微调的数据，学习和分析它，进行合适的修改。
- 指令、输入、输出都可以进行修改与增加额外的内容
- 指令中需要有明确的对象（如宠物犬，猫和狗），如果没有，请添加；比如：结核病的介绍，改为请介绍一下宠物犬的结核病。
- 修改目的是让数据更适合用于宠物犬智能诊断大模型的微调
- 可以合理地假设场景，添加输入，但这不是必要的。
- 输出需详细完整，尽可能保留原文描述。
- 每次只处理一条数据，也只返回一条数据
"""
QA_PAIRS_HUMAN_PROMPT = """
请严格按照如下格式返回结果，请勿返回任何其他提示语句与除数据外的内容（如数据已清洗：，这种提示语句）：
指令: [替换内容]
输入: [替换内容]
输出: [替换内容]

数据如下：
指令: {instruction}
输入: {input_text}
输出: {output_text}
"""
# 第二步：定义数据清洗函数
def clean_data_with_llm(instruction, input_text, output_text):
    
    messages = [
        {"role": "system", "content": QA_PAIRS_SYSTEM_PROMPT},
        {"role": "user", "content": QA_PAIRS_HUMAN_PROMPT.format(instruction=instruction, input_text = input_text, output_text = output_text)}
    ]
    response = client.chat.completions.create(model="qwen-long", messages=messages,
                                              max_tokens=1500)
    
    # 解析返回结果，确保格式正确
    cleaned_text = response.choices[0].message.content
    print(cleaned_text)
    print()
    pattern = re.compile(r"指令:\s*(.*?)\s*输入:\s*(.*?)\s*输出:\s*(.*)", re.DOTALL)
    match = pattern.search(cleaned_text)
    if match:
        cleaned_instruction = match.group(1).strip()
        cleaned_input = match.group(2).strip()
        cleaned_output = match.group(3).strip()
#         print(cleaned_instruction)
#         print(cleaned_input)
#         print(cleaned_output)
        return cleaned_instruction, cleaned_input, cleaned_output
    else:
        raise ValueError("无法解析返回的文本")

# 第三步：清洗数据并生成新的数据格式
for item in data[index:]:
    print("现在进行到第{}条数据".format(index))
    index = index + 1
    instruction = item['instruction']
    input_text = item['input']
    output_text = item['output']
    
    cleaned_instruction, cleaned_input, cleaned_output = clean_data_with_llm(instruction, input_text, output_text)
    
    cleaned_item = {
        "instruction": cleaned_instruction,
        "input": cleaned_input,
        "output": cleaned_output
    }
    cleaned_data.append(cleaned_item)

# 第四步：将结果保存为新的JSON文件
with open('cleaned_data.json', 'w', encoding='utf-8') as file:
    json.dump(cleaned_data, file, ensure_ascii=False, indent=4)


现在进行到第1723条数据
指令: 宠物犬结核病的症状和诊断方法是什么？
输入:
输出: 宠物犬结核病主要由结核分枝杆菌引起，症状可能包括持续性体重减轻、食欲不振、慢性咳嗽、乏力、发热、呼吸困难和可能出现淋巴结肿大。在诊断方面，兽医通常会进行以下步骤：

1. 临床检查：兽医会观察和触摸犬的身体，寻找可能的病灶和体征。

2. 痰液或组织样本检查：通过显微镜检查犬的痰液或活体组织切片，寻找结核分枝杆菌。

3. 结核菌素测试（PPD测试）：这是一种皮肤测试，可以检测犬体内对结核菌的反应。

4. X光和CT扫描：这些影像学检查有助于发现肺部或其他器官的病变。

5. 血液测试：虽然血液测试不能直接确认结核病，但可以帮助排除其他可能的疾病。

6. 培养测试：将样本培养在特殊培养基上，如果结核菌生长，可以确认诊断。

早期诊断和治疗至关重要，因为结核病如果不加以控制，可能会对犬的健康造成严重影响，并可能传播给人类。因此，一旦发现上述症状，应立即带宠物犬去兽医那里进行检查。

现在进行到第1724条数据
指令: 宠物犬可能患上的呼吸道疾病有哪些？
输入:
输出: 宠物犬可能患上的呼吸道疾病包括犬瘟热、犬窝咳（也称传染性喉气管炎）、犬副流感、肺炎、犬支气管炎、犬肺孢子虫病以及过敏性哮喘等。这些疾病通常会引起咳嗽、打喷嚏、呼吸急促、发热和精神不振等症状。及时的诊断和治疗对于防止病情恶化至关重要。

现在进行到第1725条数据
指令: 宠物犬可能患有的呼吸道疾病有哪些症状？
输入:
输出: 宠物犬可能患有的呼吸道疾病，如犬肺炎、犬传染性支气管炎等，常见症状包括：

1. 咳嗽：持续性或阵发性咳嗽，可能是干咳或有痰。
2. 流鼻涕：可能会有清水样、黏液样或脓性鼻涕。
3. 呼吸困难：喘息、呼吸急促或张口呼吸。
4. 发热：体温高于正常范围（通常犬的正常体温在38.5°C~39.5°C之间）。
5. 精神不振和食欲减退：患病的狗狗可能会变得疲倦，不喜欢活动，食欲下降。
6. 喉咙痛或吞咽困难：狗狗可能表现出不愿喝水或吃食的行为。
7. 声音嘶哑：咳嗽可能导致声音变化。
8. 胸部疼痛：当触摸胸部时，狗狗可能会表现出疼痛反应。

如果发现这些症状，应立即带狗狗去兽医那里进行检查和治疗，因为呼吸道疾病如果不及时处理，可能会导致严重并发症。

现在进行到第1726条数据
指令: 宠物犬

In [14]:
print(len(cleaned_data))

332


In [15]:
# 第四步：将结果保存为新的JSON文件
with open('cleaned_data_part6.json', 'w', encoding='utf-8') as file:
    json.dump(cleaned_data, file, ensure_ascii=False, indent=4)

In [26]:
cleaned_data

[{'instruction': '犬瘟热是由什么引起的，具有哪些特点？',
  'input': '犬瘟热',
  'output': '是由犬瘟热病毒引发的，其特点包括早期体温波动（双相热），类似感冒症状，随后发展成支气管炎、卡他性肺炎和胃肠炎。后期可能出现神经系统症状，如痉挛和抽搐，并可能导致鼻部和角质垫异常硬化（硬脚热病）。这是一种高传染性和可能致命的疾病。'},
 {'instruction': '犬瘟热的流行病学特点是什么？',
  'input': '',
  'output': '犬瘟热以冬春季节为主，每三年可能发生大规模流行。幼犬尤其易感，纯种犬和警犬较土种犬更易受感染且病情严重，死亡率较高。传播途径主要是病犬与健康犬的直接接触或空气飞沫传播，其鼻、眼分泌物和尿液是主要传染源。'},
 {'instruction': '犬瘟热的潜伏期和症状有哪些？',
  'input': '',
  'output': '犬瘟热的潜伏期通常为3至9天。初期症状包括发热，可持续1至3天，然后可能短暂下降，接着再次升高。初期可能表现为类似感冒的症状。随后，狗狗会出现眼部症状，如流泪、眼结膜充血和粘稠的分泌物。鼻腔变化明显，干燥且有浆液性或脓性鼻涕。初期咳嗽可能是干咳，随后变为湿咳并伴有呼吸困难。消化系统症状包括呕吐、腹泻，严重时可能导致肠套叠，造成脱水和整体健康状况恶化。如果出现神经性犬瘟，症状通常在初次症状后约10天左右显现。'},
 {'instruction': '犬瘟热病毒对犬的眼睛有哪些影响？',
  'input': '',
  'output': '犬瘟热感染可引起犬的眼睛并发症，包括结膜炎和角膜炎。其中，角膜炎常见于病程后期约15天，表现为角膜浑浊甚至形成溃疡，严重时可能导致视力丧失，甚至角膜穿孔。'},
 {'instruction': '犬瘟热的死亡率和继发症状有哪些？',
  'input': '犬瘟热是一种严重的犬类传染病，主要影响幼犬。',
  'output': '犬瘟热的死亡率非常高，特别是在幼犬中，可达80-90%。一旦感染，犬只可能会出现一系列继发症状，包括但不限于肺炎（呼吸困难）、严重腹泻导致的肠炎、以及罕见但危险的肠套叠。这些并发症会进一步威胁犬只的生命健康。因此，早期诊断和治疗至关重要。'},
 {'instruction':

In [4]:
!pip install openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.5/327.5 kB 8.4 MB/s eta 0:00:00ta 0:00:01
